<a href="https://colab.research.google.com/github/rayeeed/UFP_Project/blob/main/UFP_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Wet Model***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
#from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score,cross_val_predict
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectFromModel
from time import time
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import seaborn as sns
from sklearn.preprocessing import Normalizer,MinMaxScaler,StandardScaler,RobustScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso, BayesianRidge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_selection import RFE,RFECV
from sklearn.linear_model import (HuberRegressor,
                              	RANSACRegressor, TheilSenRegressor)

In [ ]:
!pip install shap
import shap

In [ ]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 7.6 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import KBinsDiscretizer

# Load the dataset
df = pd.read_excel('/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/model_35_wet_dry.xlsx', index_col='site_name')

# Drop unnecessary columns
df.drop(['Site_type', 'OID'], inplace=True, axis=1)

# Separate the target variable and features
y = df['Mean_wet_pnc']
X = df.drop(['Mean PNC (# / cm3)', 'Mean PM2.5 (µg/m3)','Mean_wet_pm','Mean_dry_pm','Mean_wet_pnc','Mean_dry_pnc'], axis=1)

In [ ]:
# Discretize the target variable into bins using 'quantile' strategy
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).flatten()

# Perform stratified train-test split
split = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
for train_index, test_index in split.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the distribution of bins in the train and test sets
print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))

# Output the resulting dataframes
print("X_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n", X_test.head())
print("y_test:\n", y_test.head())

Train bin distribution:
 0.0    5
1.0    6
3.0    5
4.0    6
2.0    6
Name: count, dtype: int64
Test bin distribution:
 0.0    2
1.0    1
3.0    2
2.0    1
4.0    1
Name: count, dtype: int64
X_train:
                 gsv_wall_p100  gsv_building_p100  gsv_house_p100  \
site_name                                                          
Uttara_Sector1       6.797307          32.993471        0.232041   
Azimpur              0.000000           0.000000        0.000000   
TSC_DU               0.945000          10.302500        0.190000   
Farmgate             2.947500          28.212500        0.000000   
US_Embassy           3.237500           2.430000        0.000000   

                gsv_awning_p100  gsv_sky_p100  gsv_earth_p100  gsv_tree_p100  \
site_name                                                                      
Uttara_Sector1          0.09694     19.991354         3.48166      15.913745   
Azimpur                 0.00000      0.000000         0.00000       0.000000   
TS

/tmp/ipython-input-3543545133.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
/tmp/ipython-input-3543545133.py:13: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))


In [ ]:
from scipy.stats import ks_2samp
ks_statistic, p_value = ks_2samp(y_train, y_test)

print(f"K-S statistic: {ks_statistic}")
print(f"P-value: {p_value}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print("The null hypothesis is rejected. The distributions of the train and test sets are different.")
else:
    print("The null hypothesis cannot be rejected. The distributions of the train and test sets are the same.")

K-S statistic: 0.25
P-value: 0.842454033893869
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same.


In [ ]:
# Apply Kolmogorov-Smirnov test for each feature
ks_results = {}
failed_features = []
alpha = 0.05
for feature in X.columns:
    ks_statistic, p_value = ks_2samp(X_train[feature], X_test[feature])
    ks_results[feature] = {'ks_statistic': ks_statistic, 'p_value': p_value}
    if p_value < alpha:
        failed_features.append(feature)

# Print K-S test results for each feature
for feature, result in ks_results.items():
    print(f"Feature: {feature}")
    print(f"K-S statistic: {result['ks_statistic']}")
    print(f"P-value: {result['p_value']}")
    if result['p_value'] < alpha:
        print("The null hypothesis is rejected. The distributions of the train and test sets are different for this feature.\n")
    else:
        print("The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.\n")

# Print features that failed the K-S test
if failed_features:
    print("Features that failed the K-S test (null hypothesis rejected):")
    for feature in failed_features:
        print(f"- {feature}")
else:
    print("All features passed the K-S test (null hypothesis not rejected).")

Feature: gsv_wall_p100
K-S statistic: 0.25
P-value: 0.842454033893869
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_building_p100
K-S statistic: 0.14285714285714285
P-value: 0.9996747723257569
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_house_p100
K-S statistic: 0.10714285714285714
P-value: 0.9999998512905011
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_awning_p100
K-S statistic: 0.21428571428571427
P-value: 0.9411846496106786
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_sky_p100
K-S statistic: 0.14285714285714285
P-value: 0.9996747723257569
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this featu

In [ ]:
X_train_dropped = X_train.drop(columns=failed_features)
X_test_dropped = X_test.drop(columns=failed_features)

In [ ]:
trans = StandardScaler()
X_st = trans.fit_transform(X_train_dropped)
X_st = pd.DataFrame(X_st, columns=X_train_dropped.columns, index=X_train_dropped.index)

In [ ]:
X_test_st = trans.transform(X_test_dropped)
X_test_st = pd.DataFrame(X_test_st, columns=X_test_dropped.columns, index=X_test_dropped.index)

In [ ]:
X_Pred = pd.read_excel('/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/combined_total_v4.0.xlsx', index_col='OID_')

In [ ]:
col=X_st.columns
X_Pred_f1 = X_Pred[col]
X_Pred_f2= X_Pred_f1

In [ ]:
x_pred_st= trans.transform(X_Pred_f2)
x_pred_st = pd.DataFrame(x_pred_st, columns = X_Pred_f2.columns,index=X_Pred_f2.index)

SVR

In [ ]:
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500'],
    ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500'],
    ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']
]

def evaluate_svr_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # 1. Fixed Kernel (Linear)
        kernel = 'linear'

        # 2. Optimize Hyperparameters
        C = trial.suggest_float("C", 0.01, 1000000.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.001, 10.0)

        # 3. Train and Evaluate
        svr = SVR(kernel=kernel, C=C, epsilon=epsilon)

        predicted = cross_val_predict(svr, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"SVR_Linear_RMSE_Set_{set_index}_Wet_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = SVR(
        kernel='linear',
        C=best_params['C'],
        epsilon=best_params['epsilon']
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_svr_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'wet_pnc')
        pred_filename = f"predictions_wet_pnc_set_{i}_svr.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_wet_pnc_set_{i}_svr_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'wet_pnc')
    combined_filename = "wet_pnc_svr_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:16:16,869] A new study created in memory with name: SVR_Linear_RMSE_Set_1_Wet_PNC
[I 2025-12-22 03:16:16,926] Trial 0 finished with value: 17992.64032902155 and parameters: {'C': 34.51158359801751, 'epsilon': 4.726895066314957}. Best is trial 0 with value: 17992.64032902155.
[I 2025-12-22 03:16:16,990] Trial 1 finished with value: 16254.137449154843 and parameters: {'C': 55097.165795366476, 'epsilon': 1.3038020831792492}. Best is trial 1 with value: 16254.137449154843.
[I 2025-12-22 03:16:17,044] Trial 2 finished with value: 18772.76318208743 and parameters: {'C': 6.5273330410788795, 'epsilon': 0.006381710313139052}. Best is trial 1 with value: 16254.137449154843.



Processing Feature Set 1: ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:16:17,100] Trial 3 finished with value: 11698.117166351842 and parameters: {'C': 2215.8930561470816, 'epsilon': 7.25932520354571}. Best is trial 3 with value: 11698.117166351842.
[I 2025-12-22 03:16:17,158] Trial 4 finished with value: 15130.495279217106 and parameters: {'C': 272.140381887835, 'epsilon': 5.687555448116665}. Best is trial 3 with value: 11698.117166351842.
[I 2025-12-22 03:16:17,208] Trial 5 finished with value: 16192.809359164976 and parameters: {'C': 25089.617163286497, 'epsilon': 1.2695053641113898}. Best is trial 3 with value: 11698.117166351842.
[I 2025-12-22 03:16:17,275] Trial 6 finished with value: 16379.271292700647 and parameters: {'C': 319037.4035935277, 'epsilon': 5.955941925641284}. Best is trial 3 with value: 11698.117166351842.
[I 2025-12-22 03:16:17,332] Trial 7 finished with value: 16381.035624222583 and parameters: {'C': 152884.78142270632, 'epsilon': 9.554758544141885}. Best is trial 3 with value: 11698.117166351842.
[I 2025-12-22 03:1

Set 1 Best Params: {'C': 1899.8399315993113, 'epsilon': 8.494291577367065}
Set 1 Test RMSE: 16896.8405
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500']


[I 2025-12-22 03:16:22,205] Trial 3 finished with value: 14602.251141293282 and parameters: {'C': 898.8352057634686, 'epsilon': 4.964633480699489}. Best is trial 0 with value: 12126.318634088962.
[I 2025-12-22 03:16:22,242] Trial 4 finished with value: 18933.150451248537 and parameters: {'C': 0.6183022658794, 'epsilon': 4.843938568052407}. Best is trial 0 with value: 12126.318634088962.
[I 2025-12-22 03:16:22,287] Trial 5 finished with value: 12130.634853805735 and parameters: {'C': 99957.73225136442, 'epsilon': 9.009085779019543}. Best is trial 0 with value: 12126.318634088962.
[I 2025-12-22 03:16:22,324] Trial 6 finished with value: 18954.974956763028 and parameters: {'C': 0.011546289706716124, 'epsilon': 4.574442487718399}. Best is trial 0 with value: 12126.318634088962.
[I 2025-12-22 03:16:22,361] Trial 7 finished with value: 12157.124842972622 and parameters: {'C': 3560.083072383034, 'epsilon': 0.006466978794341423}. Best is trial 0 with value: 12126.318634088962.
[I 2025-12-22 03

Set 2 Best Params: {'C': 2764.1611048102473, 'epsilon': 5.622884216077962}
Set 2 Test RMSE: 18310.3560
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:16:27,521] Trial 3 finished with value: 11673.751946885672 and parameters: {'C': 4136.728690585545, 'epsilon': 1.7988455522558058}. Best is trial 0 with value: 11511.837390814271.
[I 2025-12-22 03:16:27,625] Trial 4 finished with value: 11781.746464329908 and parameters: {'C': 416937.5127506677, 'epsilon': 4.2297259430939915}. Best is trial 0 with value: 11511.837390814271.
[I 2025-12-22 03:16:27,664] Trial 5 finished with value: 18924.116300880418 and parameters: {'C': 0.930451162757288, 'epsilon': 8.38784319535207}. Best is trial 0 with value: 11511.837390814271.
[I 2025-12-22 03:16:27,703] Trial 6 finished with value: 18954.092550887945 and parameters: {'C': 0.038595655429425016, 'epsilon': 6.484587713792797}. Best is trial 0 with value: 11511.837390814271.
[I 2025-12-22 03:16:27,772] Trial 7 finished with value: 11508.208724311671 and parameters: {'C': 12729.279454489208, 'epsilon': 9.933753818975836}. Best is trial 7 with value: 11508.208724311671.
[I 2025-12-22 0

Set 3 Best Params: {'C': 14602.33703672553, 'epsilon': 0.35194413225725873}
Set 3 Test RMSE: 18058.4851
--> Saved predictions for Set 3

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/wet_pnc_svr_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gs...   
1          2  ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_p...   
2          3  ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gs...   

                                         Best_Params    RMSE_Train  R2_Train  \
0  {'C': 1899.8399315993113, 'epsilon': 8.4942915...  10194.704509  0.699862   
1  {'C': 2764.1611048102473, 'epsilon': 5.6228842...  10674.048596  0.670974   
2  {'C': 14602.33703672553, 'epsilon': 0.35194413...   9661.776804  0.730421   

        RMSE_CV     R2_CV     RMSE_Test   R2_Test  
0  11409.579272  0.624067  16896.840492  0.5116

LR

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500'],
    ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500'],
    ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']
]

def evaluate_linear_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    # Initialize and Fit Linear Regression
    model = LinearRegression()
    model.fit(X_Train, y_train)

    # --- Calculate Metrics ---

    # 1. Training
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Testing
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_linear_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'wet_pnc')
        pred_filename = f"predictions_wet_pnc_set_{i}_linear.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_wet_pnc_set_{i}_linear_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'wet_pnc')
    combined_filename = "wet_pnc_linear_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")


Processing Feature Set 1: ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500']
Set 1 Test RMSE: 17005.5238
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500']
Set 2 Test RMSE: 17595.0373
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']
Set 3 Test RMSE: 19043.5953
--> Saved predictions for Set 3

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/wet_pnc_linear_combined_performance_metrics.csv
   Set_Index                                           Features    RMSE_Train  \
0          1  ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gs...   9937.192226   
1          2  ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_p...  10599.439203   
2          3  ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gs...   8626.164306   

   R2_Train       RMSE_CV     R2_CV     RMSE_Test   R2_T

XGBoost

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor
import optuna

# Define the new list of feature sets
feature_sets = [
    ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500'],
    ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500'],
    ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']
]

def evaluate_xgb_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameters
        learning_rate = trial.suggest_float('learning_rate', 0.005, 1.5)
        n_estimators = trial.suggest_int('n_estimators', 50, 1000)
        reg_alpha = trial.suggest_float('reg_alpha', 0.001, 100)
        reg_lambda = trial.suggest_float('reg_lambda', 0.001, 3)

        XGB = XGBRegressor(
            booster='gblinear',
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            objective="reg:squarederror",
            seed=42,
            verbosity=0
        )

        predicted = cross_val_predict(XGB, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"XGB_RMSE_Set_{set_index}_Wet_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=200)

    best_params = study.best_params

    # Rebuild final model
    model = XGBRegressor(
        booster='gblinear',
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        objective="reg:squarederror",
        seed=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_xgb_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'wet_pnc')
        pred_filename = f"predictions_wet_pnc_set_{i}_xgb.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_wet_pnc_set_{i}_xgb_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'wet_pnc')
    combined_filename = "wet_pnc_xgb_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:18:19,638] A new study created in memory with name: XGB_RMSE_Set_1_Wet_PNC



Processing Feature Set 1: ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:18:19,937] Trial 0 finished with value: 15334.409749955626 and parameters: {'learning_rate': 1.3279486839469876, 'n_estimators': 358, 'reg_alpha': 11.06036964763067, 'reg_lambda': 2.9253945420672354}. Best is trial 0 with value: 15334.409749955626.
[I 2025-12-22 03:18:20,386] Trial 1 finished with value: 14965.623849270361 and parameters: {'learning_rate': 0.8142620937044887, 'n_estimators': 602, 'reg_alpha': 15.569388471462725, 'reg_lambda': 2.5153427370540515}. Best is trial 1 with value: 14965.623849270361.
[I 2025-12-22 03:18:20,771] Trial 2 finished with value: 14517.535724199364 and parameters: {'learning_rate': 0.7999292949985932, 'n_estimators': 486, 'reg_alpha': 53.74784951586411, 'reg_lambda': 2.0779710869585983}. Best is trial 2 with value: 14517.535724199364.
[I 2025-12-22 03:18:21,190] Trial 3 finished with value: 14945.508015422545 and parameters: {'learning_rate': 0.8351988871300056, 'n_estimators': 639, 'reg_alpha': 3.2683465182435705, 'reg_lambda': 2.5

Set 1 Best Params: {'learning_rate': 0.028105189515836862, 'n_estimators': 103, 'reg_alpha': 27.481619117205575, 'reg_lambda': 0.4157348869569406}
Set 1 Test RMSE: 18040.6151
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500']


[I 2025-12-22 03:19:58,970] Trial 0 finished with value: 13821.20366032769 and parameters: {'learning_rate': 0.4754865802556681, 'n_estimators': 698, 'reg_alpha': 13.262350118420532, 'reg_lambda': 0.9971445251546003}. Best is trial 0 with value: 13821.20366032769.
[I 2025-12-22 03:19:59,173] Trial 1 finished with value: 15572.677415397502 and parameters: {'learning_rate': 0.6407933794277247, 'n_estimators': 267, 'reg_alpha': 85.82060983125973, 'reg_lambda': 2.6618064121184215}. Best is trial 0 with value: 13821.20366032769.
[I 2025-12-22 03:19:59,528] Trial 2 finished with value: 12702.328840873213 and parameters: {'learning_rate': 1.4290173800550996, 'n_estimators': 505, 'reg_alpha': 88.16751104820132, 'reg_lambda': 0.19499006302665978}. Best is trial 2 with value: 12702.328840873213.
[I 2025-12-22 03:19:59,936] Trial 3 finished with value: 14171.482840005065 and parameters: {'learning_rate': 0.4358248693507736, 'n_estimators': 682, 'reg_alpha': 61.51666831987965, 'reg_lambda': 1.2441

Set 2 Best Params: {'learning_rate': 1.4253232830586826, 'n_estimators': 355, 'reg_alpha': 68.3676759171626, 'reg_lambda': 0.05883287360515405}
Set 2 Test RMSE: 17860.1193
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:21:54,088] Trial 1 finished with value: 11299.854918075904 and parameters: {'learning_rate': 0.48752674318767725, 'n_estimators': 646, 'reg_alpha': 99.42627518652678, 'reg_lambda': 0.4624258732408691}. Best is trial 1 with value: 11299.854918075904.
[I 2025-12-22 03:21:54,710] Trial 2 finished with value: 13909.29057758165 and parameters: {'learning_rate': 0.15050935833508738, 'n_estimators': 287, 'reg_alpha': 38.05032474614709, 'reg_lambda': 2.213996304933276}. Best is trial 1 with value: 11299.854918075904.
[I 2025-12-22 03:21:57,084] Trial 3 finished with value: 14625.586398453843 and parameters: {'learning_rate': 0.4815970182667306, 'n_estimators': 633, 'reg_alpha': 30.82389926941916, 'reg_lambda': 2.9280448215890766}. Best is trial 1 with value: 11299.854918075904.
[I 2025-12-22 03:21:57,757] Trial 4 finished with value: 10884.940748811201 and parameters: {'learning_rate': 1.397009788040655, 'n_estimators': 255, 'reg_alpha': 74.6233625506346, 'reg_lambda': 0.13659

Set 3 Best Params: {'learning_rate': 0.00511961133813732, 'n_estimators': 768, 'reg_alpha': 77.0117872912898, 'reg_lambda': 0.008199723914693147}
Set 3 Test RMSE: 18803.6521
--> Saved predictions for Set 3

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/wet_pnc_xgb_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gs...   
1          2  ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_p...   
2          3  ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gs...   

                                         Best_Params    RMSE_Train  R2_Train  \
0  {'learning_rate': 0.028105189515836862, 'n_est...  10392.554422  0.688099   
1  {'learning_rate': 1.4253232830586826, 'n_estim...  10622.747238  0.674129   
2  {'learning_rate': 0.00511961133813732, 'n_esti...   8636.008805  0.784624   

        RMSE_CV     R2_CV    

Ridge Regression


In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500'],
    ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500'],
    ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']
]

def evaluate_ridge_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Ridge: Alpha
        alpha = trial.suggest_float('alpha', 0.01, 10000.0, log=True)

        model = Ridge(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"Ridge_RMSE_Set_{set_index}_Wet_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = Ridge(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_ridge_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'wet_pnc')
        pred_filename = f"predictions_wet_pnc_set_{i}_ridge.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_wet_pnc_set_{i}_ridge_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'wet_pnc')
    combined_filename = "wet_pnc_ridge_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:24:12,279] A new study created in memory with name: Ridge_RMSE_Set_1_Wet_PNC
[I 2025-12-22 03:24:12,326] Trial 0 finished with value: 15278.409859645304 and parameters: {'alpha': 1.1931933628444447}. Best is trial 0 with value: 15278.409859645304.
[I 2025-12-22 03:24:12,369] Trial 1 finished with value: 16720.30482213464 and parameters: {'alpha': 134.69438539117428}. Best is trial 0 with value: 15278.409859645304.
[I 2025-12-22 03:24:12,415] Trial 2 finished with value: 18076.292609784203 and parameters: {'alpha': 0.03459015441024074}. Best is trial 0 with value: 15278.409859645304.
[I 2025-12-22 03:24:12,458] Trial 3 finished with value: 17854.64670618122 and parameters: {'alpha': 256.7867391939977}. Best is trial 0 with value: 15278.409859645304.
[I 2025-12-22 03:24:12,499] Trial 4 finished with value: 18079.106811039077 and parameters: {'alpha': 0.03376884602096205}. Best is trial 0 with value: 15278.409859645304.



Processing Feature Set 1: ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:24:12,540] Trial 5 finished with value: 15840.39897504156 and parameters: {'alpha': 0.8856330492712019}. Best is trial 0 with value: 15278.409859645304.
[I 2025-12-22 03:24:12,583] Trial 6 finished with value: 18981.12409312924 and parameters: {'alpha': 828.9072652313065}. Best is trial 0 with value: 15278.409859645304.
[I 2025-12-22 03:24:12,628] Trial 7 finished with value: 13367.983122435773 and parameters: {'alpha': 31.978285586728045}. Best is trial 7 with value: 13367.983122435773.
[I 2025-12-22 03:24:12,667] Trial 8 finished with value: 19258.259851210714 and parameters: {'alpha': 1570.5422208316074}. Best is trial 7 with value: 13367.983122435773.
[I 2025-12-22 03:24:12,706] Trial 9 finished with value: 19459.113611343484 and parameters: {'alpha': 4119.165533284555}. Best is trial 7 with value: 13367.983122435773.
[I 2025-12-22 03:24:12,751] Trial 10 finished with value: 12439.762142120844 and parameters: {'alpha': 18.698099861050796}. Best is trial 10 with val

Set 1 Best Params: {'alpha': 9.689086356266134}
Set 1 Test RMSE: 17848.1650
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500']


[I 2025-12-22 03:24:16,886] Trial 3 finished with value: 19277.924238500767 and parameters: {'alpha': 1644.819317162376}. Best is trial 0 with value: 12643.70688170694.
[I 2025-12-22 03:24:16,928] Trial 4 finished with value: 16647.972705169042 and parameters: {'alpha': 116.82282407686328}. Best is trial 0 with value: 12643.70688170694.
[I 2025-12-22 03:24:16,972] Trial 5 finished with value: 12670.583158962652 and parameters: {'alpha': 0.19128361048893938}. Best is trial 0 with value: 12643.70688170694.
[I 2025-12-22 03:24:17,021] Trial 6 finished with value: 13069.591580505612 and parameters: {'alpha': 12.078144605273264}. Best is trial 0 with value: 12643.70688170694.
[I 2025-12-22 03:24:17,065] Trial 7 finished with value: 18737.779123528675 and parameters: {'alpha': 559.6282117945609}. Best is trial 0 with value: 12643.70688170694.
[I 2025-12-22 03:24:17,106] Trial 8 finished with value: 17283.343114577212 and parameters: {'alpha': 166.8226637771389}. Best is trial 0 with value: 1

Set 2 Best Params: {'alpha': 2.0237482440674777}
Set 2 Test RMSE: 17895.6262
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:24:21,495] Trial 2 finished with value: 19314.66547297574 and parameters: {'alpha': 2411.8396235814303}. Best is trial 1 with value: 10863.634066850687.
[I 2025-12-22 03:24:21,552] Trial 3 finished with value: 19368.540719918205 and parameters: {'alpha': 3021.5200060626767}. Best is trial 1 with value: 10863.634066850687.
[I 2025-12-22 03:24:21,622] Trial 4 finished with value: 10853.954018951163 and parameters: {'alpha': 0.3312072036246555}. Best is trial 4 with value: 10853.954018951163.
[I 2025-12-22 03:24:21,684] Trial 5 finished with value: 13585.177936819724 and parameters: {'alpha': 49.24317728941735}. Best is trial 4 with value: 10853.954018951163.
[I 2025-12-22 03:24:21,750] Trial 6 finished with value: 10863.788469228193 and parameters: {'alpha': 0.013201985850306476}. Best is trial 4 with value: 10853.954018951163.
[I 2025-12-22 03:24:21,805] Trial 7 finished with value: 10863.020339341918 and parameters: {'alpha': 0.035564490662292426}. Best is trial 4 with

Set 3 Best Params: {'alpha': 1.7935375420112907}
Set 3 Test RMSE: 18935.6851
--> Saved predictions for Set 3

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/wet_pnc_ridge_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gs...   
1          2  ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_p...   
2          3  ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gs...   

                     Best_Params    RMSE_Train  R2_Train       RMSE_CV  \
0   {'alpha': 9.689086356266134}  10242.318717  0.697052  12002.154907   
1  {'alpha': 2.0237482440674777}  10628.432880  0.673780  12614.890967   
2  {'alpha': 1.7935375420112907}   8646.486522  0.784101  10834.812363   

      R2_CV     RMSE_Test   R2_Test  
0  0.584003  17848.164960  0.455135  
1  0.540444  17895.626206  0.452233  
2  0.660988  18935.685083  0.38671

Lasso

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500'],
    ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500'],
    ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']
]

def evaluate_lasso_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Lasso: Alpha
        alpha = trial.suggest_float('alpha', 0.0001, 100.0, log=True)

        model = Lasso(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"Lasso_RMSE_Set_{set_index}_Wet_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = Lasso(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_lasso_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'wet_pnc')
        pred_filename = f"predictions_wet_pnc_set_{i}_lasso.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_wet_pnc_set_{i}_lasso_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'wet_pnc')
    combined_filename = "wet_pnc_lasso_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:28:03,944] A new study created in memory with name: Lasso_RMSE_Set_1_Wet_PNC
[I 2025-12-22 03:28:03,992] Trial 0 finished with value: 18196.19816068799 and parameters: {'alpha': 0.0029003958766038516}. Best is trial 0 with value: 18196.19816068799.
[I 2025-12-22 03:28:04,056] Trial 1 finished with value: 18195.910934270683 and parameters: {'alpha': 0.06515142065260247}. Best is trial 1 with value: 18195.910934270683.
[I 2025-12-22 03:28:04,102] Trial 2 finished with value: 18167.736961632385 and parameters: {'alpha': 6.318663575747536}. Best is trial 2 with value: 18167.736961632385.



Processing Feature Set 1: ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:28:04,149] Trial 3 finished with value: 18196.149674288154 and parameters: {'alpha': 0.013475577366825283}. Best is trial 2 with value: 18167.736961632385.
[I 2025-12-22 03:28:04,193] Trial 4 finished with value: 18184.177909620903 and parameters: {'alpha': 2.6724234575255816}. Best is trial 2 with value: 18167.736961632385.
[I 2025-12-22 03:28:04,237] Trial 5 finished with value: 18195.703488527844 and parameters: {'alpha': 0.11101350617182533}. Best is trial 2 with value: 18167.736961632385.
[I 2025-12-22 03:28:04,288] Trial 6 finished with value: 18196.187063911722 and parameters: {'alpha': 0.005326487007366621}. Best is trial 2 with value: 18167.736961632385.
[I 2025-12-22 03:28:04,330] Trial 7 finished with value: 18195.386552629217 and parameters: {'alpha': 0.17887112985751183}. Best is trial 2 with value: 18167.736961632385.
[I 2025-12-22 03:28:04,372] Trial 8 finished with value: 18196.207851516712 and parameters: {'alpha': 0.0007758289062839472}. Best is trial

Set 1 Best Params: {'alpha': 99.72804502848916}
Set 1 Test RMSE: 17082.5328
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_plant_p500']


[I 2025-12-22 03:28:09,163] Trial 2 finished with value: 12685.348285830984 and parameters: {'alpha': 0.000736327089791124}. Best is trial 1 with value: 12683.389492593249.
[I 2025-12-22 03:28:09,220] Trial 3 finished with value: 12670.545922337691 and parameters: {'alpha': 22.81504516102696}. Best is trial 3 with value: 12670.545922337691.
[I 2025-12-22 03:28:09,278] Trial 4 finished with value: 12626.496380995173 and parameters: {'alpha': 93.52963054234269}. Best is trial 4 with value: 12626.496380995173.
[I 2025-12-22 03:28:09,345] Trial 5 finished with value: 12685.002632486518 and parameters: {'alpha': 0.5956756759310172}. Best is trial 4 with value: 12626.496380995173.
[I 2025-12-22 03:28:09,405] Trial 6 finished with value: 12648.557844665984 and parameters: {'alpha': 57.63821501612428}. Best is trial 4 with value: 12626.496380995173.
[I 2025-12-22 03:28:09,461] Trial 7 finished with value: 12685.064397865688 and parameters: {'alpha': 0.5019071115014159}. Best is trial 4 with va

Set 2 Best Params: {'alpha': 99.60690041349605}
Set 2 Test RMSE: 17620.6088
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gsv_plant_p500']


[I 2025-12-22 03:28:14,748] Trial 3 finished with value: 10864.248104963805 and parameters: {'alpha': 0.0015293094705461274}. Best is trial 3 with value: 10864.248104963805.
[I 2025-12-22 03:28:14,791] Trial 4 finished with value: 10864.247591328847 and parameters: {'alpha': 0.00014347712155021015}. Best is trial 4 with value: 10864.247591328847.
[I 2025-12-22 03:28:14,834] Trial 5 finished with value: 10864.430313272029 and parameters: {'alpha': 0.6469866320884969}. Best is trial 4 with value: 10864.247591328847.
[I 2025-12-22 03:28:14,881] Trial 6 finished with value: 10864.247801074338 and parameters: {'alpha': 0.000496839763100939}. Best is trial 4 with value: 10864.247591328847.
[I 2025-12-22 03:28:14,932] Trial 7 finished with value: 10864.28421787537 and parameters: {'alpha': 0.11041320086546735}. Best is trial 4 with value: 10864.247591328847.
[I 2025-12-22 03:28:14,976] Trial 8 finished with value: 10864.897350593077 and parameters: {'alpha': 2.465024951306435}. Best is trial 

Set 3 Best Params: {'alpha': 0.018066603514050573}
Set 3 Test RMSE: 19043.6102
--> Saved predictions for Set 3

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/wet_pnc_lasso_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_150', 'gsv_bus_p150', 'RdL3_1000', 'gs...   
1          2  ['RdAll_150', 'RdL3_1000', 'RdclP_250', 'gsv_p...   
2          3  ['RdAll_150', 'gsv_car_p150', 'RdL3_1000', 'gs...   

                       Best_Params    RMSE_Train  R2_Train       RMSE_CV  \
0     {'alpha': 99.72804502848916}   9938.400580  0.714764  17754.336060   
1     {'alpha': 99.60690041349605}  10600.386837  0.675500  12622.830720   
2  {'alpha': 0.018066603514050573}   8626.164306  0.785114  10864.247088   

      R2_CV     RMSE_Test   R2_Test  
0  0.089708  17082.532840  0.500878  
1  0.539865  17620.608782  0.468940  
2  0.659144  19043.61023

# ***Dry Model***

SVR

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import KBinsDiscretizer

# Load the dataset
df = pd.read_excel('/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/model_35_wet_dry.xlsx', index_col='site_name')

# Drop unnecessary columns
df.drop(['Site_type', 'OID'], inplace=True, axis=1)

# Separate the target variable and features
y = df['Mean_dry_pnc']
X = df.drop(['Mean PNC (# / cm3)', 'Mean PM2.5 (µg/m3)','Mean_wet_pm','Mean_dry_pm','Mean_wet_pnc','Mean_dry_pnc'], axis=1)

In [ ]:
# Discretize the target variable into bins using 'quantile' strategy
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).flatten()

# Perform stratified train-test split
split = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
for train_index, test_index in split.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the distribution of bins in the train and test sets
print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))

# Output the resulting dataframes
print("X_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n", X_test.head())
print("y_test:\n", y_test.head())

Train bin distribution:
 0.0    5
1.0    6
3.0    5
4.0    6
2.0    6
Name: count, dtype: int64
Test bin distribution:
 0.0    2
1.0    1
3.0    2
2.0    1
4.0    1
Name: count, dtype: int64
X_train:
                  gsv_wall_p100  gsv_building_p100  gsv_house_p100  \
site_name                                                           
Uttara_Sector1        6.797307          32.993471        0.232041   
Azimpur               0.000000           0.000000        0.000000   
Shangsad_bhaban       0.100000           2.592500        0.000000   
Gulisthan             0.412500          26.647500        0.000000   
TSC_DU                0.945000          10.302500        0.190000   

                 gsv_awning_p100  gsv_sky_p100  gsv_earth_p100  gsv_tree_p100  \
site_name                                                                       
Uttara_Sector1           0.09694     19.991354         3.48166      15.913745   
Azimpur                  0.00000      0.000000         0.00000       0.0

/tmp/ipython-input-3543545133.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
/tmp/ipython-input-3543545133.py:13: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))


In [ ]:
# Discretize the target variable into bins using 'quantile' strategy
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).flatten()

# Perform stratified train-test split
split = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
for train_index, test_index in split.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the distribution of bins in the train and test sets
print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))

# Output the resulting dataframes
print("X_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n", X_test.head())
print("y_test:\n", y_test.head())

Train bin distribution:
 0.0    5
1.0    6
3.0    5
4.0    6
2.0    6
Name: count, dtype: int64
Test bin distribution:
 0.0    2
1.0    1
3.0    2
2.0    1
4.0    1
Name: count, dtype: int64
X_train:
                  gsv_wall_p100  gsv_building_p100  gsv_house_p100  \
site_name                                                           
Uttara_Sector1        6.797307          32.993471        0.232041   
Azimpur               0.000000           0.000000        0.000000   
Shangsad_bhaban       0.100000           2.592500        0.000000   
Gulisthan             0.412500          26.647500        0.000000   
TSC_DU                0.945000          10.302500        0.190000   

                 gsv_awning_p100  gsv_sky_p100  gsv_earth_p100  gsv_tree_p100  \
site_name                                                                       
Uttara_Sector1           0.09694     19.991354         3.48166      15.913745   
Azimpur                  0.00000      0.000000         0.00000       0.0

/tmp/ipython-input-3543545133.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
/tmp/ipython-input-3543545133.py:13: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))


In [ ]:
# Apply Kolmogorov-Smirnov test for each feature
ks_results = {}
failed_features = []
alpha = 0.05
for feature in X.columns:
    ks_statistic, p_value = ks_2samp(X_train[feature], X_test[feature])
    ks_results[feature] = {'ks_statistic': ks_statistic, 'p_value': p_value}
    if p_value < alpha:
        failed_features.append(feature)

# Print K-S test results for each feature
for feature, result in ks_results.items():
    print(f"Feature: {feature}")
    print(f"K-S statistic: {result['ks_statistic']}")
    print(f"P-value: {result['p_value']}")
    if result['p_value'] < alpha:
        print("The null hypothesis is rejected. The distributions of the train and test sets are different for this feature.\n")
    else:
        print("The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.\n")

# Print features that failed the K-S test
if failed_features:
    print("Features that failed the K-S test (null hypothesis rejected):")
    for feature in failed_features:
        print(f"- {feature}")
else:
    print("All features passed the K-S test (null hypothesis not rejected).")

Feature: gsv_wall_p100
K-S statistic: 0.39285714285714285
P-value: 0.3111686782104894
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_building_p100
K-S statistic: 0.21428571428571427
P-value: 0.9411846496106786
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_house_p100
K-S statistic: 0.10714285714285714
P-value: 0.9999998512905011
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_awning_p100
K-S statistic: 0.2857142857142857
P-value: 0.7050303962215889
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_sky_p100
K-S statistic: 0.42857142857142855
P-value: 0.21839938017880833
The null hypothesis cannot be rejected. The distributions of the train and test sets are the sam

In [ ]:
X_train_dropped = X_train.drop(columns=failed_features)
X_test_dropped = X_test.drop(columns=failed_features)

trans = StandardScaler()
X_st = trans.fit_transform(X_train_dropped)
X_st = pd.DataFrame(X_st, columns=X_train_dropped.columns, index=X_train_dropped.index)

X_test_st = trans.transform(X_test_dropped)
X_test_st = pd.DataFrame(X_test_st, columns=X_test_dropped.columns, index=X_test_dropped.index)


In [ ]:
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150'],
    ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']
]

def evaluate_svr_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # 1. Fixed Kernel (Linear)
        kernel = 'linear'

        # 2. Optimize Hyperparameters
        C = trial.suggest_float("C", 0.01, 1000000.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.001, 10.0)

        # 3. Train and Evaluate
        svr = SVR(kernel=kernel, C=C, epsilon=epsilon)

        predicted = cross_val_predict(svr, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"SVR_Linear_RMSE_Set_{set_index}_Dry_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = SVR(
        kernel='linear',
        C=best_params['C'],
        epsilon=best_params['epsilon']
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_svr_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'dry_pnc')
        pred_filename = f"predictions_dry_pnc_set_{i}_svr.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_pnc_set_{i}_svr_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'dry_pnc')
    combined_filename = "dry_pnc_svr_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:39:15,415] A new study created in memory with name: SVR_Linear_RMSE_Set_1_Dry_PNC
[I 2025-12-22 03:39:15,465] Trial 0 finished with value: 14765.317255548785 and parameters: {'C': 438363.4899303271, 'epsilon': 7.902847026159915}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,521] Trial 1 finished with value: 14818.393713170304 and parameters: {'C': 44977.211042489245, 'epsilon': 5.774307941538927}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,558] Trial 2 finished with value: 21779.157238957814 and parameters: {'C': 1.0857710294964957, 'epsilon': 6.187897273807767}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,596] Trial 3 finished with value: 21583.1528294969 and parameters: {'C': 13.468288943944215, 'epsilon': 1.4041410131955245}. Best is trial 0 with value: 14765.317255548785.



Processing Feature Set 1: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250']


[I 2025-12-22 03:39:15,633] Trial 4 finished with value: 21795.78051001923 and parameters: {'C': 0.04155151128627439, 'epsilon': 8.93197686972559}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,671] Trial 5 finished with value: 21740.815933722854 and parameters: {'C': 3.4977387854156485, 'epsilon': 4.472866992711257}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,714] Trial 6 finished with value: 21790.452645285968 and parameters: {'C': 0.3761313055029848, 'epsilon': 0.38680797922292615}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,752] Trial 7 finished with value: 21145.822322675493 and parameters: {'C': 41.595039326815126, 'epsilon': 7.250470282510699}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 03:39:15,788] Trial 8 finished with value: 15132.868777031741 and parameters: {'C': 11017.655308499316, 'epsilon': 5.867887699879175}. Best is trial 0 with value: 14765.317255548785.
[I 2025-12-22 

Set 1 Best Params: {'C': 954337.6746367671, 'epsilon': 9.996340737139018}
Set 1 Test RMSE: 13895.9200
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250']


[I 2025-12-22 03:39:21,416] Trial 4 finished with value: 19510.107159948166 and parameters: {'C': 783323.0929426111, 'epsilon': 7.312046781651514}. Best is trial 1 with value: 15856.923604614725.
[I 2025-12-22 03:39:21,455] Trial 5 finished with value: 21670.535615725923 and parameters: {'C': 16.00536123090335, 'epsilon': 9.911335716754758}. Best is trial 1 with value: 15856.923604614725.
[I 2025-12-22 03:39:21,496] Trial 6 finished with value: 14910.326425032174 and parameters: {'C': 4699.143542238841, 'epsilon': 2.5533503036293963}. Best is trial 6 with value: 14910.326425032174.
[I 2025-12-22 03:39:21,537] Trial 7 finished with value: 19506.469967186826 and parameters: {'C': 62806.69576736777, 'epsilon': 7.710302161067975}. Best is trial 6 with value: 14910.326425032174.
[I 2025-12-22 03:39:21,578] Trial 8 finished with value: 18865.358858931544 and parameters: {'C': 36644.66566483623, 'epsilon': 9.825871461450125}. Best is trial 6 with value: 14910.326425032174.
[I 2025-12-22 03:39

Set 2 Best Params: {'C': 4699.143542238841, 'epsilon': 2.5533503036293963}
Set 2 Test RMSE: 14220.5077
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150']


[I 2025-12-22 03:39:26,210] Trial 2 finished with value: 21777.39526298227 and parameters: {'C': 1.501309418856707, 'epsilon': 0.19202426071353706}. Best is trial 1 with value: 16133.023985198659.
[I 2025-12-22 03:39:26,263] Trial 3 finished with value: 15819.895105402458 and parameters: {'C': 22969.235544954627, 'epsilon': 9.912726292741782}. Best is trial 3 with value: 15819.895105402458.
[I 2025-12-22 03:39:26,314] Trial 4 finished with value: 21792.252717141568 and parameters: {'C': 0.3300119018828854, 'epsilon': 2.934969948361559}. Best is trial 3 with value: 15819.895105402458.
[I 2025-12-22 03:39:26,369] Trial 5 finished with value: 20415.975794290574 and parameters: {'C': 81993.5134894773, 'epsilon': 6.59004641329981}. Best is trial 3 with value: 15819.895105402458.
[I 2025-12-22 03:39:26,426] Trial 6 finished with value: 20668.779020520717 and parameters: {'C': 106218.84036971093, 'epsilon': 4.138090557600312}. Best is trial 3 with value: 15819.895105402458.
[I 2025-12-22 03:3

Set 3 Best Params: {'C': 8404.356270908629, 'epsilon': 7.747110216117655}
Set 3 Test RMSE: 14389.5934
--> Saved predictions for Set 3

Processing Feature Set 4: ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']


[I 2025-12-22 03:39:31,813] Trial 4 finished with value: 16933.144678317247 and parameters: {'C': 1328.8376962656057, 'epsilon': 0.8967476261687682}. Best is trial 3 with value: 15066.486456170293.
[I 2025-12-22 03:39:31,850] Trial 5 finished with value: 16181.412537617247 and parameters: {'C': 453.90900063300603, 'epsilon': 9.495455835845677}. Best is trial 3 with value: 15066.486456170293.
[I 2025-12-22 03:39:31,886] Trial 6 finished with value: 21794.27457511003 and parameters: {'C': 0.10803739252763593, 'epsilon': 4.404371004549913}. Best is trial 3 with value: 15066.486456170293.
[I 2025-12-22 03:39:31,926] Trial 7 finished with value: 21795.161909840528 and parameters: {'C': 0.06381200018319044, 'epsilon': 4.911835927863213}. Best is trial 3 with value: 15066.486456170293.
[I 2025-12-22 03:39:31,963] Trial 8 finished with value: 21470.58185469056 and parameters: {'C': 16.344901751218103, 'epsilon': 0.9649936567277698}. Best is trial 3 with value: 15066.486456170293.
[I 2025-12-22

Set 4 Best Params: {'C': 282375.57678351103, 'epsilon': 9.994650910453926}
Set 4 Test RMSE: 15905.2873
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_pnc_svr_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
1          2  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
2          3  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
3          4  ['RdAll_250', 'gsv_house_p750', 'RdclT_250', '...   

                                         Best_Params    RMSE_Train  R2_Train  \
0  {'C': 954337.6746367671, 'epsilon': 9.99634073...  11151.641762  0.705520   
1  {'C': 4699.143542238841, 'epsilon': 2.55335030...  12330.890101  0.639946   
2  {'C': 8404.356270908629, 'epsilon': 7.74711021...  12299.465835  0.641779   
3  {'C': 282375.57678351103, 'epsi

Linear regression

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150'],
    ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']
]

def evaluate_linear_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    # Initialize and Fit Linear Regression
    model = LinearRegression()
    model.fit(X_Train, y_train)

    # --- Calculate Metrics ---

    # 1. Training
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Testing
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_linear_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'dry_pnc')
        pred_filename = f"predictions_dry_pnc_set_{i}_linear.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_pnc_set_{i}_linear_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'dry_pnc')
    combined_filename = "dry_pnc_linear_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")


Processing Feature Set 1: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250']
Set 1 Test RMSE: 13988.2427
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250']
Set 2 Test RMSE: 14883.0452
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150']
Set 3 Test RMSE: 15102.4337
--> Saved predictions for Set 3

Processing Feature Set 4: ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']
Set 4 Test RMSE: 15100.8806
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_pnc_linear_combined_performance_metrics.csv
   Set_Index                                           Features    RMSE_Train  \
0          1  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...  10546.593998   
1          2  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...  11940

xgboost

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor
import optuna

# Define the new list of feature sets
feature_sets = [
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150'],
    ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']
]

def evaluate_xgb_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameters
        learning_rate = trial.suggest_float('learning_rate', 0.005, 1.5)
        n_estimators = trial.suggest_int('n_estimators', 50, 1000)
        reg_alpha = trial.suggest_float('reg_alpha', 0.001, 100)
        reg_lambda = trial.suggest_float('reg_lambda', 0.001, 3)

        XGB = XGBRegressor(
            booster='gblinear',
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            objective="reg:squarederror",
            seed=42,
            verbosity=0
        )

        predicted = cross_val_predict(XGB, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"XGB_RMSE_Set_{set_index}_Dry_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=200)

    best_params = study.best_params

    # Rebuild final model
    model = XGBRegressor(
        booster='gblinear',
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        objective="reg:squarederror",
        seed=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_xgb_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'dry_pnc')
        pred_filename = f"predictions_dry_pnc_set_{i}_xgb.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_pnc_set_{i}_xgb_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'dry_pnc')
    combined_filename = "dry_pnc_xgb_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:44:04,029] A new study created in memory with name: XGB_RMSE_Set_1_Dry_PNC



Processing Feature Set 1: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250']


[I 2025-12-22 03:44:05,007] Trial 0 finished with value: 16513.146257602162 and parameters: {'learning_rate': 0.14677128508307805, 'n_estimators': 690, 'reg_alpha': 33.18721972265228, 'reg_lambda': 1.6517973382436135}. Best is trial 0 with value: 16513.146257602162.
[I 2025-12-22 03:44:05,209] Trial 1 finished with value: 16492.77445084745 and parameters: {'learning_rate': 0.3556318738144577, 'n_estimators': 196, 'reg_alpha': 2.369800310875974, 'reg_lambda': 1.6509560959425025}. Best is trial 1 with value: 16492.77445084745.
[I 2025-12-22 03:44:05,370] Trial 2 finished with value: 13280.873053414683 and parameters: {'learning_rate': 1.3424791397763094, 'n_estimators': 166, 'reg_alpha': 74.70599339046703, 'reg_lambda': 0.2595433367404749}. Best is trial 2 with value: 13280.873053414683.
[I 2025-12-22 03:44:05,771] Trial 3 finished with value: 17185.086132174634 and parameters: {'learning_rate': 0.9411014702405096, 'n_estimators': 665, 'reg_alpha': 88.79604476861599, 'reg_lambda': 2.0950

Set 1 Best Params: {'learning_rate': 1.19375715774217, 'n_estimators': 999, 'reg_alpha': 65.10571799039903, 'reg_lambda': 0.15164810626820655}
Set 1 Test RMSE: 13817.0313
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250']


[I 2025-12-22 03:46:04,568] Trial 1 finished with value: 14938.27963728631 and parameters: {'learning_rate': 1.4707517008062083, 'n_estimators': 165, 'reg_alpha': 98.57866448107706, 'reg_lambda': 0.4924207051295282}. Best is trial 1 with value: 14938.27963728631.
[I 2025-12-22 03:46:05,090] Trial 2 finished with value: 16442.056279810517 and parameters: {'learning_rate': 0.7994473302067091, 'n_estimators': 863, 'reg_alpha': 56.802520118020404, 'reg_lambda': 1.2372553044294095}. Best is trial 1 with value: 14938.27963728631.
[I 2025-12-22 03:46:05,553] Trial 3 finished with value: 14705.309424065501 and parameters: {'learning_rate': 1.2806627493347449, 'n_estimators': 829, 'reg_alpha': 85.66676054241132, 'reg_lambda': 0.3447147444847158}. Best is trial 3 with value: 14705.309424065501.
[I 2025-12-22 03:46:05,845] Trial 4 finished with value: 14707.37910769841 and parameters: {'learning_rate': 0.07910143752194418, 'n_estimators': 447, 'reg_alpha': 86.50577915764043, 'reg_lambda': 0.34648

Set 2 Best Params: {'learning_rate': 1.0734460272054798, 'n_estimators': 458, 'reg_alpha': 2.3416176964610846, 'reg_lambda': 0.28882704574604273}
Set 2 Test RMSE: 14476.3455
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150']


[I 2025-12-22 03:47:38,968] Trial 0 finished with value: 16140.061419812859 and parameters: {'learning_rate': 0.00860029292204572, 'n_estimators': 948, 'reg_alpha': 86.51228764282457, 'reg_lambda': 1.3111080517966975}. Best is trial 0 with value: 16140.061419812859.
[I 2025-12-22 03:47:39,382] Trial 1 finished with value: 17905.028054259114 and parameters: {'learning_rate': 0.28865963900966973, 'n_estimators': 607, 'reg_alpha': 84.78200276894123, 'reg_lambda': 2.7957065768806006}. Best is trial 0 with value: 16140.061419812859.
[I 2025-12-22 03:47:39,708] Trial 2 finished with value: 17284.360996572337 and parameters: {'learning_rate': 0.875829722990093, 'n_estimators': 483, 'reg_alpha': 25.37723993310922, 'reg_lambda': 2.1726300996113}. Best is trial 0 with value: 16140.061419812859.
[I 2025-12-22 03:47:39,842] Trial 3 finished with value: 16825.321008017185 and parameters: {'learning_rate': 0.9745193509920653, 'n_estimators': 140, 'reg_alpha': 45.052264087495, 'reg_lambda': 1.7842767

Set 3 Best Params: {'learning_rate': 0.533433164779772, 'n_estimators': 761, 'reg_alpha': 18.803217447525814, 'reg_lambda': 0.2730466352504209}
Set 3 Test RMSE: 14530.1169
--> Saved predictions for Set 3

Processing Feature Set 4: ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']


[I 2025-12-22 03:49:17,024] Trial 0 finished with value: 14588.148979267706 and parameters: {'learning_rate': 0.18091612668033935, 'n_estimators': 794, 'reg_alpha': 38.77484668980562, 'reg_lambda': 0.39963643534135945}. Best is trial 0 with value: 14588.148979267706.
[I 2025-12-22 03:49:17,556] Trial 1 finished with value: 17854.802914816562 and parameters: {'learning_rate': 0.039909278025043265, 'n_estimators': 933, 'reg_alpha': 67.79483306304569, 'reg_lambda': 2.3393766178540587}. Best is trial 0 with value: 14588.148979267706.
[I 2025-12-22 03:49:18,024] Trial 2 finished with value: 17803.48620660464 and parameters: {'learning_rate': 1.4115342363914303, 'n_estimators': 768, 'reg_alpha': 19.419833242566135, 'reg_lambda': 2.3109871396917785}. Best is trial 0 with value: 14588.148979267706.
[I 2025-12-22 03:49:18,180] Trial 3 finished with value: 14562.804513279922 and parameters: {'learning_rate': 0.9172987892787721, 'n_estimators': 130, 'reg_alpha': 24.500481781783012, 'reg_lambda': 

Set 4 Best Params: {'learning_rate': 1.0075399387382118, 'n_estimators': 379, 'reg_alpha': 1.7448707612222094, 'reg_lambda': 0.22637135057801158}
Set 4 Test RMSE: 14657.5917
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_pnc_xgb_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
1          2  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
2          3  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
3          4  ['RdAll_250', 'gsv_house_p750', 'RdclT_250', '...   

                                         Best_Params    RMSE_Train  R2_Train  \
0  {'learning_rate': 1.19375715774217, 'n_estimat...  10750.426935  0.726328   
1  {'learning_rate': 1.0734460272054798, 'n_estim...  12338.082995  0.639526   
2  {'learning_rate': 0.533433164779772, 'n_

ridge

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150'],
    ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']
]

def evaluate_ridge_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Ridge: Alpha
        alpha = trial.suggest_float('alpha', 0.01, 10000.0, log=True)

        model = Ridge(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"Ridge_RMSE_Set_{set_index}_Dry_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = Ridge(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_ridge_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'dry_pnc')
        pred_filename = f"predictions_dry_pnc_set_{i}_ridge.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_pnc_set_{i}_ridge_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'dry_pnc')
    combined_filename = "dry_pnc_ridge_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:50:42,232] A new study created in memory with name: Ridge_RMSE_Set_1_Dry_PNC
[I 2025-12-22 03:50:42,276] Trial 0 finished with value: 21633.552879884403 and parameters: {'alpha': 4491.979903444922}. Best is trial 0 with value: 21633.552879884403.
[I 2025-12-22 03:50:42,323] Trial 1 finished with value: 13832.785846039935 and parameters: {'alpha': 0.03815108015065117}. Best is trial 1 with value: 13832.785846039935.
[I 2025-12-22 03:50:42,374] Trial 2 finished with value: 13834.012838760245 and parameters: {'alpha': 0.03561307313170726}. Best is trial 1 with value: 13832.785846039935.
[I 2025-12-22 03:50:42,413] Trial 3 finished with value: 21435.07499840221 and parameters: {'alpha': 1363.9659724931698}. Best is trial 1 with value: 13832.785846039935.



Processing Feature Set 1: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250']


[I 2025-12-22 03:50:42,460] Trial 4 finished with value: 21676.89702970976 and parameters: {'alpha': 8817.13192327302}. Best is trial 1 with value: 13832.785846039935.
[I 2025-12-22 03:50:42,504] Trial 5 finished with value: 13818.477636074986 and parameters: {'alpha': 0.06804121508766128}. Best is trial 5 with value: 13818.477636074986.
[I 2025-12-22 03:50:42,561] Trial 6 finished with value: 13308.679272300165 and parameters: {'alpha': 7.4876079845508015}. Best is trial 6 with value: 13308.679272300165.
[I 2025-12-22 03:50:42,609] Trial 7 finished with value: 13316.345856842934 and parameters: {'alpha': 7.579800990648288}. Best is trial 6 with value: 13308.679272300165.
[I 2025-12-22 03:50:42,649] Trial 8 finished with value: 13594.578727221638 and parameters: {'alpha': 10.432499169156635}. Best is trial 6 with value: 13308.679272300165.
[I 2025-12-22 03:50:42,691] Trial 9 finished with value: 13410.506843288329 and parameters: {'alpha': 8.627515385831073}. Best is trial 6 with value

Set 1 Best Params: {'alpha': 4.1712715888740055}
Set 1 Test RMSE: 13817.6728
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250']


[I 2025-12-22 03:50:47,294] Trial 2 finished with value: 14700.692851322288 and parameters: {'alpha': 9.28409804673592}. Best is trial 2 with value: 14700.692851322288.
[I 2025-12-22 03:50:47,349] Trial 3 finished with value: 16888.301310899504 and parameters: {'alpha': 0.08446508595186167}. Best is trial 2 with value: 14700.692851322288.
[I 2025-12-22 03:50:47,401] Trial 4 finished with value: 16572.900078951738 and parameters: {'alpha': 0.3821085581274216}. Best is trial 2 with value: 14700.692851322288.
[I 2025-12-22 03:50:47,466] Trial 5 finished with value: 16494.291142519574 and parameters: {'alpha': 32.136827189955234}. Best is trial 2 with value: 14700.692851322288.
[I 2025-12-22 03:50:47,522] Trial 6 finished with value: 14987.915783685336 and parameters: {'alpha': 3.4754145971074735}. Best is trial 2 with value: 14700.692851322288.
[I 2025-12-22 03:50:47,577] Trial 7 finished with value: 16153.333593254885 and parameters: {'alpha': 27.4850734259009}. Best is trial 2 with valu

Set 2 Best Params: {'alpha': 7.311182300063435}
Set 2 Test RMSE: 14491.2900
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150']


[I 2025-12-22 03:50:52,962] Trial 4 finished with value: 14765.757906544583 and parameters: {'alpha': 2.5448721983652396}. Best is trial 4 with value: 14765.757906544583.
[I 2025-12-22 03:50:53,007] Trial 5 finished with value: 14999.68454024224 and parameters: {'alpha': 1.946921084829737}. Best is trial 4 with value: 14765.757906544583.
[I 2025-12-22 03:50:53,054] Trial 6 finished with value: 16132.404294022266 and parameters: {'alpha': 0.32570067266770686}. Best is trial 4 with value: 14765.757906544583.
[I 2025-12-22 03:50:53,098] Trial 7 finished with value: 18441.89958237482 and parameters: {'alpha': 90.17086578442753}. Best is trial 4 with value: 14765.757906544583.
[I 2025-12-22 03:50:53,158] Trial 8 finished with value: 20447.460923048842 and parameters: {'alpha': 301.72771966851735}. Best is trial 4 with value: 14765.757906544583.
[I 2025-12-22 03:50:53,201] Trial 9 finished with value: 21017.346543683376 and parameters: {'alpha': 580.8335508625796}. Best is trial 4 with value

Set 3 Best Params: {'alpha': 6.981591335816354}
Set 3 Test RMSE: 14574.2705
--> Saved predictions for Set 3

Processing Feature Set 4: ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']


[I 2025-12-22 03:50:57,514] Trial 3 finished with value: 21666.14350938468 and parameters: {'alpha': 6550.372224085615}. Best is trial 2 with value: 14407.743229616746.
[I 2025-12-22 03:50:57,558] Trial 4 finished with value: 15131.456663186518 and parameters: {'alpha': 0.06281460509356818}. Best is trial 2 with value: 14407.743229616746.
[I 2025-12-22 03:50:57,600] Trial 5 finished with value: 14496.212386796695 and parameters: {'alpha': 8.793091845500113}. Best is trial 2 with value: 14407.743229616746.
[I 2025-12-22 03:50:57,647] Trial 6 finished with value: 15464.905588241581 and parameters: {'alpha': 20.562627270487884}. Best is trial 2 with value: 14407.743229616746.
[I 2025-12-22 03:50:57,691] Trial 7 finished with value: 20936.065655752813 and parameters: {'alpha': 434.52023681313045}. Best is trial 2 with value: 14407.743229616746.
[I 2025-12-22 03:50:57,733] Trial 8 finished with value: 15119.391609595948 and parameters: {'alpha': 0.09885611485829279}. Best is trial 2 with va

Set 4 Best Params: {'alpha': 5.815100001768663}
Set 4 Test RMSE: 14676.6234
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_pnc_ridge_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
1          2  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
2          3  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
3          4  ['RdAll_250', 'gsv_house_p750', 'RdclT_250', '...   

                     Best_Params    RMSE_Train  R2_Train       RMSE_CV  \
0  {'alpha': 4.1712715888740055}  10724.670229  0.727638  13146.544551   
1   {'alpha': 7.311182300063435}  12277.552687  0.643054  14651.401287   
2   {'alpha': 6.981591335816354}  12391.758448  0.636383  14230.714827   
3   {'alpha': 5.815100001768663}  12149.791659  0.650444  14405.299393   

      R2

lasso

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the new list of feature sets
feature_sets = [
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250'],
    ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150'],
    ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']
]

def evaluate_lasso_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Lasso: Alpha
        alpha = trial.suggest_float('alpha', 0.0001, 100.0, log=True)

        model = Lasso(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"Lasso_RMSE_Set_{set_index}_Dry_PNC"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = Lasso(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_lasso_model(features, i)

        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions (using 'dry_pnc')
        pred_filename = f"predictions_dry_pnc_set_{i}_lasso.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_pnc_set_{i}_lasso_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file (using 'dry_pnc')
    combined_filename = "dry_pnc_lasso_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-22 03:51:02,205] A new study created in memory with name: Lasso_RMSE_Set_1_Dry_PNC
[I 2025-12-22 03:51:02,270] Trial 0 finished with value: 13851.431078888985 and parameters: {'alpha': 0.002444985555192423}. Best is trial 0 with value: 13851.431078888985.



Processing Feature Set 1: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'RdclT_250']


[I 2025-12-22 03:51:02,336] Trial 1 finished with value: 13851.383369010553 and parameters: {'alpha': 0.10169447101801792}. Best is trial 1 with value: 13851.383369010553.
[I 2025-12-22 03:51:02,401] Trial 2 finished with value: 13851.427566433598 and parameters: {'alpha': 0.00915242762539934}. Best is trial 1 with value: 13851.383369010553.
[I 2025-12-22 03:51:02,457] Trial 3 finished with value: 13851.240845786824 and parameters: {'alpha': 0.40138349583005023}. Best is trial 3 with value: 13851.240845786824.
[I 2025-12-22 03:51:02,515] Trial 4 finished with value: 13851.432134221379 and parameters: {'alpha': 0.00015780724604970677}. Best is trial 3 with value: 13851.240845786824.
[I 2025-12-22 03:51:02,583] Trial 5 finished with value: 13851.296189031096 and parameters: {'alpha': 0.3131593577296548}. Best is trial 3 with value: 13851.240845786824.
[I 2025-12-22 03:51:02,644] Trial 6 finished with value: 13851.20968246283 and parameters: {'alpha': 0.6866101100615851}. Best is trial 6 

Set 1 Best Params: {'alpha': 99.99381741062588}
Set 1 Test RMSE: 13972.6815
--> Saved predictions for Set 1

Processing Feature Set 2: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_wall_p250']


[I 2025-12-22 03:51:08,116] Trial 3 finished with value: 16987.01089960468 and parameters: {'alpha': 0.00012800212034465153}. Best is trial 2 with value: 16881.84463461576.
[I 2025-12-22 03:51:08,159] Trial 4 finished with value: 16987.010056512496 and parameters: {'alpha': 0.0005804214322997425}. Best is trial 2 with value: 16881.84463461576.
[I 2025-12-22 03:51:08,199] Trial 5 finished with value: 16987.010914122417 and parameters: {'alpha': 0.0001126343778295077}. Best is trial 2 with value: 16881.84463461576.
[I 2025-12-22 03:51:08,243] Trial 6 finished with value: 16987.010589983252 and parameters: {'alpha': 0.00028380604927736485}. Best is trial 2 with value: 16881.84463461576.
[I 2025-12-22 03:51:08,295] Trial 7 finished with value: 16986.962112993613 and parameters: {'alpha': 0.02628514759797173}. Best is trial 2 with value: 16881.84463461576.
[I 2025-12-22 03:51:08,345] Trial 8 finished with value: 16804.46803052816 and parameters: {'alpha': 95.9880775784667}. Best is trial 8 

Set 2 Best Params: {'alpha': 99.89881932816957}
Set 2 Test RMSE: 14855.2384
--> Saved predictions for Set 2

Processing Feature Set 3: ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', 'gsv_car_p150']


[I 2025-12-22 03:51:13,128] Trial 1 finished with value: 16499.95369906777 and parameters: {'alpha': 0.00013678034827402177}. Best is trial 0 with value: 16459.795030190355.
[I 2025-12-22 03:51:13,209] Trial 2 finished with value: 16499.945915182776 and parameters: {'alpha': 0.003608849956914394}. Best is trial 0 with value: 16459.795030190355.
[I 2025-12-22 03:51:13,264] Trial 3 finished with value: 16499.95363138201 and parameters: {'alpha': 0.00016578071194299336}. Best is trial 0 with value: 16459.795030190355.
[I 2025-12-22 03:51:13,309] Trial 4 finished with value: 16499.941314219417 and parameters: {'alpha': 0.005764864738550004}. Best is trial 0 with value: 16459.795030190355.
[I 2025-12-22 03:51:13,361] Trial 5 finished with value: 16499.95115864716 and parameters: {'alpha': 0.0012581384227543415}. Best is trial 0 with value: 16459.795030190355.
[I 2025-12-22 03:51:13,403] Trial 6 finished with value: 16488.782439853094 and parameters: {'alpha': 4.774655549299971}. Best is tri

Set 3 Best Params: {'alpha': 99.7870085276119}
Set 3 Test RMSE: 15055.5580
--> Saved predictions for Set 3

Processing Feature Set 4: ['RdAll_250', 'gsv_house_p750', 'RdclT_250', 'gsv_car_p150']


[I 2025-12-22 03:51:19,232] Trial 2 finished with value: 15147.063573828875 and parameters: {'alpha': 33.06308668865051}. Best is trial 2 with value: 15147.063573828875.
[I 2025-12-22 03:51:19,303] Trial 3 finished with value: 15152.85676972856 and parameters: {'alpha': 0.0006187648360149193}. Best is trial 2 with value: 15147.063573828875.
[I 2025-12-22 03:51:19,366] Trial 4 finished with value: 15152.49491419907 and parameters: {'alpha': 2.308879107063772}. Best is trial 2 with value: 15147.063573828875.
[I 2025-12-22 03:51:19,429] Trial 5 finished with value: 15152.85676194274 and parameters: {'alpha': 0.0006576855721749748}. Best is trial 2 with value: 15147.063573828875.
[I 2025-12-22 03:51:19,470] Trial 6 finished with value: 15152.856830607292 and parameters: {'alpha': 0.00025160723302781606}. Best is trial 2 with value: 15147.063573828875.
[I 2025-12-22 03:51:19,511] Trial 7 finished with value: 15138.552230120831 and parameters: {'alpha': 99.30410801256397}. Best is trial 7 wi

Set 4 Best Params: {'alpha': 99.45102834206585}
Set 4 Test RMSE: 15071.0145
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_pnc_lasso_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
1          2  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
2          3  ['RdAll_250', 'RdL3_1000', 'gsv_house_p750', '...   
3          4  ['RdAll_250', 'gsv_house_p750', 'RdclT_250', '...   

                    Best_Params    RMSE_Train  R2_Train       RMSE_CV  \
0  {'alpha': 99.99381741062588}  10548.299205  0.736522  13798.337887   
1  {'alpha': 99.89881932816957}  11941.736971  0.662314  16797.310427   
2   {'alpha': 99.7870085276119}  12121.553317  0.652067  16275.100157   
3  {'alpha': 99.45102834206585}  11862.819775  0.666762  15138.539198   

      R2_CV  